In [ ]:
###############################################################
# IFCS 2026 DATA CHALLENGE
# Gaussian Mixture Clustering Pipeline
###############################################################

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

from sklearn.decomposition import TruncatedSVD
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import linkage, dendrogram

###############################################################
# Load
###############################################################

df = pd.read_csv("train.csv")

###############################################################
# Drop ID column
###############################################################

id_cols = [c for c in df.columns if "id" in c.lower()]

df = df.drop(columns=id_cols)

###############################################################
# Guess target
###############################################################

target_candidates = [
    c for c in df.columns
    if "distress" in c.lower()
]

target = target_candidates[0] if len(target_candidates) else None

###############################################################
# Detect categorical columns
###############################################################

categorical = df.select_dtypes(
    include=["object","category"]
).columns.tolist()

if target in categorical:
    categorical.remove(target)

###############################################################
# Numerical columns
###############################################################

numerical = [
    c for c in df.columns
    if c not in categorical + ([target] if target else [])
]

###############################################################
# Log transform highly skewed variables
###############################################################

for c in numerical:

    if (df[c] >= 0).all():

        skew = df[c].skew()

        if abs(skew) > 1:
            df[c] = np.log1p(df[c])

###############################################################
# Preprocessing
###############################################################

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, numerical),
    ("cat", cat_pipe, categorical)
])

X = preprocessor.fit_transform(df)

###############################################################
# PCA
###############################################################

# Choose a reasonable number of latent dimensions
n_components = min(50, X.shape[1] - 1)

pca = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

X_pca = pca.fit_transform(X)

print("Explained variance:",
      pca.explained_variance_ratio_.sum())

###############################################################
# Automatic model selection using BIC
###############################################################

bic = []
models = []

for k in range(2, 16):
    print("Fitting GMM with", k, "components...")

    gmm = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=42,
        n_init=10
    )

    gmm.fit(X_pca)

    bic.append(gmm.bic(X_pca))
    models.append(gmm)

best = np.argmin(bic)

gmm = models[best]

labels = gmm.predict(X_pca)

df["Cluster"] = labels

print("\nBest number of clusters:", gmm.n_components)

###############################################################
# Metrics
###############################################################

sil = silhouette_score(X_pca, labels)

print("Silhouette:", sil)

###############################################################
# BIC curve
###############################################################

plt.figure(figsize=(8,5))

plt.plot(
    range(2,16),
    bic,
    marker="o"
)

plt.xlabel("Number of clusters")
plt.ylabel("BIC")
plt.title("Gaussian Mixture Model Selection")

plt.show()

###############################################################
# PCA Scatter
###############################################################

plt.figure(figsize=(9,7))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=labels,
    s=10,
    cmap="tab20"
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Clusters in PCA space")

plt.show()

###############################################################
# Cluster sizes
###############################################################

plt.figure(figsize=(7,4))

df.Cluster.value_counts().sort_index().plot.bar()

plt.title("Cluster sizes")

plt.show()

###############################################################
# Cluster profile
###############################################################

profile = df.groupby("Cluster")[numerical].median()

print(profile)

###############################################################
# Heatmap
###############################################################

scaled = (
    profile-profile.mean()
)/profile.std()

plt.figure(figsize=(14,8))

sns.heatmap(
    scaled,
    cmap="RdBu_r",
    center=0
)

plt.title("Median Financial Profile")

plt.show()

###############################################################
# Top distinguishing variables
###############################################################

overall = df[numerical].median()

for c in sorted(df.Cluster.unique()):

    print("\n")
    print("="*70)
    print("Cluster", c)

    local = df[df.Cluster==c][numerical].median()

    diff = (local-overall)

    diff = diff.reindex(
        diff.abs().sort_values(
            ascending=False
        ).index
    )

    print(diff.head(15))

###############################################################
# Distress rate
###############################################################

if target:

    plt.figure(figsize=(7,4))

    (
        df.groupby("Cluster")[target]
        .mean()
        .sort_values()
        .plot.bar()
    )

    plt.ylabel("Financial distress")

    plt.title("Financial Distress by Cluster")

    plt.show()

###############################################################
# Province analysis
###############################################################

province_cols = [
    c for c in df.columns
    if "province" in c.lower()
]

if province_cols:

    province = province_cols[0]

    table = pd.crosstab(
        df.Cluster,
        df[province],
        normalize="index"
    )

    plt.figure(figsize=(16,5))

    sns.heatmap(
        table,
        cmap="viridis"
    )

    plt.title("Province Composition")

    plt.show()

###############################################################
# Sector analysis
###############################################################

sector_cols = [
    c for c in df.columns
    if "sector" in c.lower()
]

if sector_cols:

    sector = sector_cols[0]

    table = pd.crosstab(
        df.Cluster,
        df[sector],
        normalize="index"
    )

    plt.figure(figsize=(12,6))

    sns.heatmap(
        table,
        cmap="Blues"
    )

    plt.title("Sector Composition")

    plt.show()

###############################################################
# Hierarchical clustering of clusters
###############################################################

centroids = []

for c in sorted(df.Cluster.unique()):

    centroids.append(
        X_pca[df.Cluster==c].mean(axis=0)
    )

centroids = np.vstack(centroids)

Z = linkage(
    centroids,
    method="ward"
)

plt.figure(figsize=(10,5))

dendrogram(
    Z,
    labels=[f"C{i}" for i in sorted(df.Cluster.unique())]
)

plt.title("Relationships Between Clusters")

plt.show()

###############################################################
# PCA Loadings
###############################################################

loadings = pd.DataFrame(
    pca.components_.T,
    index=preprocessor.get_feature_names_out()
)

print("\nMost important variables for PC1")

print(
    loadings[0]
    .abs()
    .sort_values(ascending=False)
    .head(20)
)

###############################################################
# Export results
###############################################################

df.to_csv("clustered_dataset.csv", index=False)

profile.to_csv("cluster_profiles.csv")

print("\nDone.")

KeyboardInterrupt: 